# Lección 1.3 — Tipos de aprendizaje: supervisado, no supervisado, semi-supervisado, auto-supervisado y por refuerzo

> **Módulo 1 · ¿Qué es aprender para una máquina?** · Nivel 1 — Cero
> ⏱️ Tiempo estimado: 70–85 min (lección con ejercicios intercalados ≈ 50 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 1.2 (vocabulario: ejemplo, característica, etiqueta, modelo).

### 🎯 Objetivos
Al terminar esta lección podrás:
- Distinguir los **cinco tipos de aprendizaje** según **qué información** recibe el modelo.
- Ver en código cada uno: supervisado, no supervisado (agrupar), semi-supervisado, auto-supervisado (fabricar etiquetas del propio texto) y por refuerzo (aprender por recompensas).
- Elegir el tipo de aprendizaje adecuado para un problema de negocio.
- Entender por qué el aprendizaje **auto-supervisado** hizo posibles los LLMs.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números o palabras.

## ▶️ Preparación

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.semi_supervised import LabelSpreading
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

plt.rcParams["figure.figsize"] = (6, 4)
print("✅ Todo listo")

---
## 1. 📖 La intuición: cinco maneras de aprender a cocinar

| Cómo aprendes | Tipo de aprendizaje |
|---|---|
| Un chef te muestra 100 platos y te dice el nombre de cada uno | **Supervisado**: ejemplos **con** respuesta |
| Te dejan solo en una despensa y ordenas los ingredientes en grupos que tengan sentido | **No supervisado**: ejemplos **sin** respuesta |
| El chef etiqueta 10 platos y tú tienes que arreglártelas con otros 1,000 sin etiquetar | **Semi-supervisado**: **pocas** respuestas y muchos datos sin ellas |
| Lees 1,000 recetas tapando el último ingrediente y tratas de adivinarlo | **Auto-supervisado**: las respuestas salen **de los propios datos** |
| Cocinas, los comensales te dan propina (o no), y ajustas | **Por refuerzo**: **recompensas** por tus acciones |

La pregunta clave para distinguirlos es: **¿qué información recibe el modelo para aprender?**

### ✋ Ejercicio 1 — Primera intuición
¿Qué tipo de aprendizaje describe cada situación?
1. Un bebé aprende a caminar: se cae, lo vuelve a intentar, y cada paso logrado es una pequeña victoria.
2. Un estudiante practica con exámenes anteriores que traen las respuestas al final.
3. Un bibliotecario recibe 10,000 libros sin clasificar y los organiza en estantes por temas que él mismo inventa.

✍️ *Mis respuestas:* 1 → ___ · 2 → ___ · 3 → ___

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Por refuerzo** (prueba acciones y recibe "recompensas": avanzar, no caerse).
2. **Supervisado** (ejemplos con la respuesta correcta).
3. **No supervisado** (nadie le dijo qué temas existen; descubre la estructura).
</details>

---
## 2. 📖 Supervisado: aprender con las respuestas

Es lo que hiciste en las lecciones 1.1 y 1.2: cada ejemplo trae su **etiqueta** y el modelo aprende a pasar de características a etiqueta. Es, con diferencia, el tipo **más usado** en la industria.

```
(características, etiqueta)  ×  muchos ejemplos   ──►  modelo  ──►  etiqueta para casos nuevos
```

Ejemplos: predecir precios, detectar spam, diagnosticar con radiografías, estimar riesgo de impago.

**Su gran limitación:** alguien tiene que **etiquetar** los datos, y eso suele ser caro (médicos revisando radiografías, personas clasificando miles de correos…).

### ✋ Ejercicio 2
Una aseguradora tiene 50,000 fotos de choques. Quiere un modelo que estime el costo de reparación mirando la foto. ¿Qué etiqueta necesita para cada foto? ¿Quién podría ponerla y por qué sería caro?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Necesita el **costo real de reparación** de cada choque. Puede venir del historial (lo que realmente se pagó), lo cual es barato si existe. Si no existe, un **perito** tendría que revisar cada foto: 50,000 fotos × unos minutos cada una = meses de trabajo de expertos. El costo de etiquetar es la razón de ser de los tipos de aprendizaje que vienen a continuación.
</details>

---
## 3. 📖 No supervisado: descubrir estructura sin respuestas

Aquí **no hay etiqueta**. El modelo recibe solo características y busca **estructura**: grupos, patrones, rarezas.

El caso típico es la **segmentación de clientes**. Una tienda tiene el gasto y la frecuencia de visita de 300 clientes, pero nadie le ha dicho "qué tipos de clientes" existen.

In [ ]:
rng_c = np.random.default_rng(7)
clientes = pd.DataFrame({
    "visitas_al_mes": np.concatenate([rng_c.normal(2, 1, 100), rng_c.normal(8, 1.5, 100), rng_c.normal(15, 2, 100)]),
    "gasto_mensual":  np.concatenate([rng_c.normal(250, 40, 100), rng_c.normal(900, 80, 100), rng_c.normal(550, 50, 100)]),
}).clip(lower=0).round(0)

plt.scatter(clientes.visitas_al_mes, clientes.gasto_mensual, color="gray", s=15)
plt.xlabel("Visitas al mes"); plt.ylabel("Gasto mensual ($)"); plt.title("Clientes SIN etiqueta"); plt.show()

### 🔮 Ejercicio 3 — ¿Cuántos grupos ves?
Mirando la gráfica: ¿cuántos **tipos de cliente** distingues? Descríbelos con palabras ("los que vienen poco y gastan…").

✍️ *Mis grupos:*

Ahora le pedimos a **K-means** (un algoritmo de agrupamiento, Lección 8.2) que encuentre **3 grupos** por sí solo. Nadie le dice qué significa cada uno.

In [ ]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=0)      # n_clusters es un HIPERPARÁMETRO: lo eliges tú
clientes["grupo"] = kmeans.fit_predict(clientes[["visitas_al_mes", "gasto_mensual"]])

plt.scatter(clientes.visitas_al_mes, clientes.gasto_mensual, c=clientes.grupo, cmap="viridis", s=15)
plt.xlabel("Visitas al mes"); plt.ylabel("Gasto mensual ($)"); plt.title("Grupos encontrados por K-means"); plt.show()

print(clientes.groupby("grupo")[["visitas_al_mes", "gasto_mensual"]].mean().round(0).assign(clientes=clientes.grupo.value_counts()))

<details><summary><b>👀 Ver respuesta</b></summary>

Hay **3 grupos** (de unos 100 clientes cada uno), y K-means los encuentra sin que nadie se los diga:
- **Ocasionales:** ~2 visitas al mes y ~250 $ de gasto.
- **Grandes compradores:** ~8 visitas y ~900 $: no vienen tanto, pero gastan mucho.
- **Frecuentes de gasto medio:** ~15 visitas y ~550 $: vienen muchísimo, pero gastan poco cada vez.

⚠️ Fíjate en un detalle: el algoritmo solo dice "grupo 0, 1, 2". **Los nombres ("ocasionales", "grandes compradores") los pones tú**, interpretando los promedios. En el aprendizaje no supervisado, la interpretación humana es parte del trabajo.
</details>

### 📖 Otros usos del aprendizaje no supervisado
- **Agrupamiento** (*clustering*): segmentar clientes, agrupar noticias por tema (Lecciones 8.2–8.3).
- **Reducción de dimensionalidad**: resumir 100 columnas en 2 para poder dibujarlas (Lección 8.4).
- **Detección de anomalías**: encontrar transacciones "raras" sin ejemplos previos de fraude (Lección 8.6).

### ✋ Ejercicio 4
Un banco quiere detectar fraudes, pero solo tiene **3** casos confirmados de fraude entre 2 millones de transacciones. ¿Por qué el aprendizaje supervisado lo tendría difícil? ¿Qué haría un enfoque no supervisado?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Con solo 3 ejemplos de fraude, un modelo supervisado casi no tiene de qué aprender (y siempre diría "no es fraude", acertando el 99.9998 %… sin servir para nada; Lección 2.7). Un enfoque **no supervisado** aprende cómo son las transacciones **normales** y marca como sospechosas las que se salen del patrón (montos, horarios o lugares inusuales), aunque nunca haya visto ese tipo de fraude.
</details>

---
## 4. 📖 Semi-supervisado: pocas etiquetas, muchos datos

Situación muy común: tienes **muchísimos** datos, pero solo **unos pocos** etiquetados (porque etiquetar es caro). El aprendizaje **semi-supervisado** aprovecha también los datos **sin etiqueta**: si dos ejemplos se parecen mucho, probablemente tengan la misma etiqueta, así que las etiquetas se pueden "contagiar" a los vecinos.

Vamos a comprobarlo con los dígitos escritos a mano: de 1,347 imágenes de entrenamiento, **solo 50 tienen etiqueta**.

In [ ]:
digitos = load_digits()
X_tr, X_te, y_tr, y_te = train_test_split(digitos.data, digitos.target, test_size=450, random_state=0)

CON_ETIQUETA = 50                                   # ✏️ prueba 20, 50 y 200
rng = np.random.default_rng(0)
elegidas = rng.choice(len(X_tr), CON_ETIQUETA, replace=False)

# Etiquetas "parciales": -1 significa "sin etiqueta"
y_parcial = np.full(len(y_tr), -1)
y_parcial[elegidas] = y_tr[elegidas]
print(f"Imágenes con etiqueta: {(y_parcial != -1).sum()} · sin etiqueta: {(y_parcial == -1).sum()}")

### 🔮 Ejercicio 5 — ¿Sirven los datos sin etiqueta?
Vamos a comparar dos modelos, examinados con las mismas 450 imágenes:
- **Solo supervisado:** aprende únicamente de las 50 imágenes etiquetadas.
- **Semi-supervisado:** usa las 50 etiquetadas **y** las 1,297 sin etiqueta.

¿Cuál crees que acertará más? ¿Por mucho o por poco?

✍️ *Mi predicción:*

In [ ]:
solo_supervisado = KNeighborsClassifier(n_neighbors=1).fit(X_tr[elegidas], y_tr[elegidas])
semi_supervisado = LabelSpreading(kernel="knn", n_neighbors=7).fit(X_tr, y_parcial)

print(f"Solo supervisado (50 imágenes)              : {solo_supervisado.score(X_te, y_te):.1%}")
print(f"Semi-supervisado (50 + {len(X_tr) - CON_ETIQUETA} sin etiqueta)  : {semi_supervisado.score(X_te, y_te):.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

**Solo supervisado: 81.6 % · semi-supervisado: 94.2 %.** ¡Más de 12 puntos de mejora sin etiquetar ni una imagen más! Los datos sin etiqueta revelan la "forma" de cada grupo de dígitos y las 50 etiquetas se contagian a sus vecinos parecidos.

✏️ Si cambias `CON_ETIQUETA` a 200, los dos mejoran y la ventaja se reduce: cuantas más etiquetas tienes, menos falta hace el truco.
</details>

---
## 5. 📖 Auto-supervisado: el truco detrás de los LLMs

¿Y si **no** tienes etiquetas, pero tampoco quieres solo agrupar? El aprendizaje **auto-supervisado** **fabrica las etiquetas a partir de los propios datos**: tapa una parte y le pide al modelo que la adivine.

Con texto, la tarea más famosa es **predecir la siguiente palabra**. Cualquier texto del mundo se convierte, gratis, en millones de ejemplos etiquetados:

In [ ]:
texto = "el gato duerme en el sofá y el perro duerme en la alfombra"

palabras = texto.split()
CONTEXTO = 2                                   # ✏️ cuántas palabras mira hacia atrás
pares = [(" ".join(palabras[i - CONTEXTO:i]), palabras[i]) for i in range(CONTEXTO, len(palabras))]

pd.DataFrame(pares, columns=["características (lo que ve)", "etiqueta (lo que debe adivinar)"])

### ✋ Ejercicio 6 — Fabrica tú las etiquetas
1. ¿Cuántos ejemplos etiquetados salieron de una frase de 13 palabras? ¿Quién los etiquetó?
2. ✏️ Cambia el `texto` por una frase tuya y `CONTEXTO` por 3. ¿Cuántos ejemplos salen ahora?
3. Si hicieras esto con todos los libros de una biblioteca, ¿cuántos ejemplos tendrías?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **11 ejemplos** (13 palabras − 2 de contexto) y **nadie** los etiquetó a mano: la etiqueta es simplemente la palabra que venía después en el texto.
2. Con `CONTEXTO = 3`, una frase de *n* palabras da *n* − 3 ejemplos.
3. **Miles de millones.** Esta es la idea clave detrás de los LLMs (GPT, Claude, Gemini…): se pre-entrenan con billones de palabras de texto adivinando la siguiente palabra, **sin que ninguna persona etiquete nada** (Lección 13.5). Después, se afinan con un poco de aprendizaje supervisado y por refuerzo.

> 📝 Con imágenes se hace algo parecido: tapar un trozo de la foto y pedir que lo reconstruya, o pedir que reconozca que dos recortes son de la misma foto.
</details>

### 🔮 Ejercicio 7 — Un mini predictor de la siguiente palabra
La celda "entrena" con los pares de arriba (usando `CONTEXTO = 1`, la palabra anterior) contando qué palabra suele seguir a cada una. ¿Qué predecirá después de **"duerme"**? ¿Y después de **"el"**?

✍️ *Mis predicciones:*

In [ ]:
from collections import Counter, defaultdict

siguientes = defaultdict(Counter)
for anterior, siguiente in zip(palabras, palabras[1:]):     # pares (palabra, palabra siguiente)
    siguientes[anterior][siguiente] += 1

for palabra in ["duerme", "el", "en"]:
    print(f"Después de '{palabra}' → {siguientes[palabra].most_common()}")

<details><summary><b>👀 Ver respuesta</b></summary>

- Después de **"duerme"** → siempre **"en"** (2 de 2 veces).
- Después de **"el"** → empate: **"gato"**, **"sofá"** y **"perro"**, una vez cada una. El modelo no puede saber cuál es, porque solo mira una palabra atrás.
- Después de **"en"** → "el" o "la".

Por eso los LLMs miran **miles** de palabras hacia atrás, no una: con más contexto ("el perro duerme en ___") la predicción se vuelve mucho más precisa. Esa capacidad de usar contexto largo es lo que aportó el Transformer (Módulo 12).
</details>

---
## 6. 📖 Por refuerzo: aprender probando y recibiendo recompensas

Aquí no hay una tabla de ejemplos. Hay un **agente** que toma **acciones** en un **entorno** y recibe **recompensas** (o castigos). Su objetivo: aprender qué acciones dan más recompensa a la larga.

```
          acción
  AGENTE ─────────►  ENTORNO
     ▲                  │
     └──── recompensa ──┘
```

Ejemplo: una app de comida tiene **3 promociones** y quiere saber cuál hace que la gente compre más. Al principio **no sabe** cuál es mejor; tiene que **probarlas**. Cada vez que muestra una promoción, el cliente compra (recompensa = 1) o no (recompensa = 0).

El dilema central es **explorar o explotar**:
- **Explotar:** mostrar la promoción que *hasta ahora* parece mejor.
- **Explorar:** probar otras, por si alguna es mejor y no lo sabemos.

Una estrategia simple (*épsilon-codiciosa*): con probabilidad **ε** (épsilon) explora al azar; el resto del tiempo explota. **ε es un hiperparámetro** (Lección 14.2).

In [ ]:
def simular_app(epsilon, clientes=1000, semilla=0):
    rng = np.random.default_rng(semilla)
    prob_compra_real = [0.3, 0.5, 0.7]          # el agente NO conoce estos números
    estimacion = np.zeros(3)                      # lo que el agente cree de cada promoción
    veces = np.zeros(3)
    compras = 0
    for _ in range(clientes):
        if rng.random() < epsilon:
            promo = rng.integers(3)               # explorar: una al azar
        else:
            promo = int(np.argmax(estimacion))    # explotar: la que parece mejor
        compra = int(rng.random() < prob_compra_real[promo])        # recompensa del entorno
        veces[promo] += 1
        estimacion[promo] += (compra - estimacion[promo]) / veces[promo]   # actualizar lo que cree
        compras += compra
    return compras, estimacion.round(2), veces.astype(int)

compras, estimacion, veces = simular_app(epsilon=0.1)
print(f"Compras logradas con 1,000 clientes: {compras}")
print(f"Lo que el agente cree de cada promoción: {estimacion.tolist()}  (la verdad: [0.3, 0.5, 0.7])")
print(f"Veces que mostró cada promoción: {veces.tolist()}")

### 🔮 Ejercicio 8 — ¿Cuánto explorar?
Vamos a comparar tres agentes:
- **ε = 0:** nunca explora (siempre muestra la que parece mejor).
- **ε = 0.1:** explora el 10 % del tiempo.
- **ε = 1:** siempre explora (elige al azar).

¿Cuál conseguirá **más compras**? ¿Cuál menos? Apuesta antes de ejecutar.

✍️ *Mis predicciones:*

In [ ]:
for eps in [0, 0.1, 1]:                   # ✏️ prueba otros valores, como 0.01 o 0.3
    compras, estimacion, veces = simular_app(eps)
    print(f"ε = {eps:<4} → compras: {compras:>3} · veces por promoción: {veces.tolist()}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **ε = 0 → 304 compras.** Se quedó atascado en la primera promoción (la peor, 30 %). Como empezó con todas a 0 y nunca exploró, **jamás descubrió** que las otras eran mejores.
- **ε = 0.1 → 677 compras.** Exploró lo justo para descubrir que la tercera era la mejor y la mostró 926 de 1,000 veces.
- **ε = 1 → 481 compras.** Explora siempre: aprende bien cuánto vale cada promoción, pero **nunca aprovecha** lo aprendido.

El equilibrio entre explorar y explotar es el corazón del aprendizaje por refuerzo, y aparece en la vida real: ¿pido mi plato favorito o pruebo uno nuevo? Así aprendieron AlphaGo (Lección 14.5) y, en parte, los asistentes como ChatGPT (RLHF, Lección 13.5).
</details>

---
## 7. 📖 Los cinco tipos, lado a lado

| Tipo | ¿Qué recibe el modelo? | Pregunta típica | Ejemplo |
|---|---|---|---|
| **Supervisado** | Ejemplos **con** etiqueta | "¿Cuál es la respuesta para este caso?" | Spam, precio de una casa |
| **No supervisado** | Ejemplos **sin** etiqueta | "¿Qué estructura hay en estos datos?" | Segmentar clientes, anomalías |
| **Semi-supervisado** | **Pocos** con etiqueta + **muchos** sin | Igual que supervisado, ahorrando etiquetas | Clasificar fotos con pocas etiquetadas |
| **Auto-supervisado** | Datos sin etiqueta, de los que **se fabrican** etiquetas | "¿Qué parte falta?" | Pre-entrenar LLMs (siguiente palabra) |
| **Por refuerzo** | **Recompensas** por sus acciones | "¿Qué hago para ganar más a la larga?" | Juegos, robots, promociones |

> 📝 En sistemas reales se **combinan**. Un LLM como ChatGPT se pre-entrena de forma **auto-supervisada**, luego se ajusta con ejemplos **supervisados** (preguntas con buenas respuestas) y finalmente con **refuerzo** a partir de las preferencias de personas (RLHF).

---
## 8. 🚨 Errores comunes y señales de alerta

| 🚨 Error | Qué pasa | Qué hacer |
|---|---|---|
| Pensar que el no supervisado "descubre la verdad" | Los grupos dependen de las características elegidas y de hiperparámetros como el número de grupos | Valida que los grupos tengan **sentido de negocio** |
| Dar por hecho que hay etiquetas | Descubres tarde que nadie registró el resultado que quieres predecir | Antes de empezar, pregunta: **¿de dónde sale la etiqueta?** |
| Usar refuerzo cuando basta supervisado | Más complejo, más lento y más arriesgado | El refuerzo es para **decisiones secuenciales** con recompensa; si tienes la respuesta correcta para cada caso, usa supervisado |
| Nunca explorar | El sistema se atasca en una opción mediocre (ε = 0) | Reservar algo de exploración |

### 🔮 Ejercicio 9 — ¿De dónde sale la etiqueta?
Una empresa quiere predecir "qué clientes están **satisfechos**". Tiene compras, visitas y reclamos, pero **nunca preguntó** a los clientes si estaban satisfechos. ¿Puede usar aprendizaje supervisado ya mismo? ¿Qué opciones tiene?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**No directamente:** falta la etiqueta. Opciones:
1. **Conseguir etiquetas:** encuestar a una muestra de clientes (y quizá usar **semi-supervisado** para aprovechar al resto).
2. **Usar una etiqueta indirecta** (*proxy*): por ejemplo, "volvió a comprar en 3 meses" o "dejó una reseña de 4–5 estrellas". Ojo: no es exactamente "satisfacción", hay que asumirlo conscientemente.
3. **No supervisado:** agrupar clientes por comportamiento y ver si algún grupo parece insatisfecho (muchos reclamos, compras en caída).
</details>

---
## 9. 🧠 Resumen en 5 puntos
1. Lo que distingue a los tipos de aprendizaje es **qué información recibe** el modelo.
2. **Supervisado** (con etiquetas) es el más usado; su cuello de botella es el **costo de etiquetar**.
3. **No supervisado** busca estructura sin etiquetas; la **interpretación** de los grupos la pones tú.
4. **Semi-supervisado** y **auto-supervisado** reducen la dependencia de etiquetas humanas; el auto-supervisado es la base de los **LLMs**.
5. **Por refuerzo** aprende de **recompensas**, equilibrando **explorar** y **explotar**.

---
## 10. 🏋️ Ejercicios finales

### 10.1 Repaso rápido (opción múltiple)

**1.** Tienes 10,000 correos, cada uno marcado como "urgente" o "no urgente". ¿Qué tipo de aprendizaje usarías para clasificar correos nuevos?
- a) Supervisado · b) No supervisado · c) Por refuerzo · d) Auto-supervisado

**2.** ¿Qué tipo de aprendizaje usa K-means?
- a) Supervisado · b) No supervisado · c) Por refuerzo · d) Semi-supervisado

**3.** ¿Cómo se consiguen las etiquetas en el pre-entrenamiento de un LLM?
- a) Miles de personas etiquetan cada frase · b) Se tapa la siguiente palabra del propio texto y se pide adivinarla · c) Con recompensas de usuarios · d) No usa etiquetas de ningún tipo

**4.** En el Ejercicio 8, ¿por qué el agente con ε = 0 consiguió tan pocas compras?
- a) Exploró demasiado · b) Nunca exploró y se quedó con la primera opción · c) Las promociones eran malas · d) Por un error del código

**5.** Tienes 1 millón de fotos de productos y solo 500 están clasificadas por categoría. ¿Qué enfoque aprovecha mejor la situación?
- a) Supervisado con las 500 y tirar el resto · b) Semi-supervisado · c) Por refuerzo · d) Ninguno

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **a)** · 2. **b)** · 3. **b)** (por eso decimos que es **auto**-supervisado: hay etiquetas, pero salen de los propios datos) · 4. **b)** · 5. **b)** (y en la práctica moderna, también se usarían modelos pre-entrenados de forma auto-supervisada; Lección 10.5).
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** En la segmentación de clientes de la Sección 3 le pedimos a K-means **3** grupos. ¿Qué crees que pasaría si le pidiéramos 2? ¿Y 10? ¿Cómo decidirías cuántos grupos usar si no pudieras ver la gráfica?

**B. Caso de análisis 💼** Netflix quiere mejorar sus recomendaciones. Propón **un uso** para cada uno de estos tipos: supervisado, no supervisado y por refuerzo.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

**A.** Con **2** grupos, K-means juntaría dos de los tipos de cliente en uno (perdiendo información). Con **10**, partiría los grupos naturales en trozos sin mucho sentido de negocio. El número de grupos es un **hiperparámetro**: se elige con medidas como el **método del codo** o la **silueta** (Lección 8.2) y, sobre todo, preguntando si los grupos resultantes **sirven para tomar decisiones**.

**B.** Un ejemplo:
- **Supervisado:** predecir si verás una película completa (etiqueta: la terminaste o no), a partir de tu historial.
- **No supervisado:** agrupar películas parecidas o usuarios con gustos parecidos, sin etiquetas.
- **Por refuerzo:** decidir qué mostrar en la pantalla de inicio, explorando de vez en cuando títulos nuevos y aprendiendo de lo que eliges (recompensa).
</details>

### 10.3 🧩 Práctica integradora: la tienda de té 🍵

Una tienda online de té tiene datos de 300 clientes. Vas a aplicar **dos** tipos de aprendizaje a los **mismos** datos y ver qué responde cada uno. 4 pasos cortos; revisa cada respuesta antes de seguir.

**Paso 1 ▶️ — Los datos**

In [ ]:
rng_te = np.random.default_rng(3)
tienda = pd.DataFrame({
    "pedidos_al_año": np.concatenate([rng_te.normal(3, 1, 150), rng_te.normal(12, 2, 150)]).clip(1).round(),
    "gasto_medio":    np.concatenate([rng_te.normal(25, 5, 150), rng_te.normal(40, 8, 150)]).clip(5).round(),
})
# ¿Se suscribió al "club del té" (1) o no (0)? Depende sobre todo de cuántos pedidos hace.
tienda["suscrito"] = (rng_te.random(300) < 1 / (1 + np.exp(-(tienda.pedidos_al_año - 8)))).astype(int)
print(f"{len(tienda)} clientes · {tienda.suscrito.mean():.0%} suscritos")
tienda.head()

**Paso 2 ✋ — Dos preguntas, dos tipos de aprendizaje.** La dueña tiene dos preguntas:
- **P1:** "¿Qué tipos de clientes tengo?"
- **P2:** "¿Este cliente nuevo se suscribirá al club?"

¿Qué tipo de aprendizaje responde cada una? ¿Qué columnas usarías como características en cada caso? ¿Hay etiqueta?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **P1 → no supervisado** (agrupamiento). Características: `pedidos_al_año` y `gasto_medio`. **Sin** etiqueta.
- **P2 → supervisado** (clasificación). Características: `pedidos_al_año` y `gasto_medio`. Etiqueta: `suscrito`.

Mismos datos, preguntas distintas → tipos de aprendizaje distintos. **La pregunta de negocio manda**, no los datos.
</details>

**Paso 3 ▶️ — P1 con K-means.** Ejecuta y describe los grupos.

In [ ]:
GRUPOS = 2   # ✏️ prueba también 3
km_te = KMeans(n_clusters=GRUPOS, n_init=10, random_state=0)
tienda["grupo"] = km_te.fit_predict(tienda[["pedidos_al_año", "gasto_medio"]])
print(tienda.groupby("grupo")[["pedidos_al_año", "gasto_medio", "suscrito"]].mean().round(2)
      .assign(clientes=tienda.grupo.value_counts()))

<details><summary><b>👀 Ver respuesta</b></summary>

Con 2 grupos aparecen dos perfiles claros:
- **Ocasionales** (165 clientes): ~4 pedidos al año y ~25 $ por pedido. Pocos están suscritos (**10 %**).
- **Fieles** (135 clientes): ~12 pedidos al año y ~43 $ por pedido. Casi todos están suscritos (**94 %**).

Fíjate: K-means **no usó** la columna `suscrito`, y aun así los grupos que encontró explican muy bien quién se suscribe. Al revisar los grupos con columnas que el algoritmo no vio, les estás dando **sentido de negocio**.
</details>

**Paso 4 🔮 — P2 con un árbol.** Entrenamos un árbol de decisión pequeño para predecir `suscrito`. Antes de ejecutar: ¿qué característica crees que usará en la **primera pregunta**? ¿Qué predecirá para un cliente con **10 pedidos** al año y 30 $ de gasto medio?

✍️ *Mis predicciones:*

In [ ]:
arbol_te = DecisionTreeClassifier(max_depth=2, random_state=0)
arbol_te.fit(tienda[["pedidos_al_año", "gasto_medio"]], tienda["suscrito"])
print(export_text(arbol_te, feature_names=["pedidos_al_año", "gasto_medio"], class_names=["no", "sí"]))

nuevo = pd.DataFrame({"pedidos_al_año": [10], "gasto_medio": [30]})     # ✏️ prueba otros clientes
print("¿Se suscribirá?", "sí" if arbol_te.predict(nuevo)[0] == 1 else "no",
      f"(probabilidad {arbol_te.predict_proba(nuevo)[0][1]:.0%})")

<details><summary><b>👀 Ver respuesta</b></summary>

La primera pregunta es sobre **`pedidos_al_año`** (≤ 7): es la característica que más separa a los suscritos de los que no. (Fíjate en que las segundas preguntas no cambian la respuesta: en cada lado, las dos hojas dicen lo mismo. Lo que sí cambia es **la probabilidad**.) Para el cliente con 10 pedidos predice **sí**, con una probabilidad del **81 %**.

**Conclusión de la práctica:** con los **mismos datos**, el no supervisado te dio un **mapa** de tus clientes y el supervisado una **predicción** para cada cliente nuevo. Son herramientas complementarias.
</details>

---
## 📚 Para profundizar
- 🎮 **Interactivo:** *Naftali Harris — Visualizing K-Means Clustering* (busca "visualizing k-means"): mira cómo se mueven los grupos paso a paso.
- 📺 **Video:** *AlphaGo* (documental, 2017): aprendizaje por refuerzo en acción.
- 📘 **Lectura:** Yann LeCun, la metáfora del "pastel": el aprendizaje auto-supervisado es el bizcocho, el supervisado el glaseado y el refuerzo la cereza (busca "LeCun cake analogy").

---
⬅️ **Anterior:** [1.2 Vocabulario esencial: dato, característica, etiqueta, modelo, predicción, parámetro e hiperparámetro](1.2-vocabulario-esencial.ipynb) · ➡️ **Siguiente:** [1.4 Tipos de problema: regresión, clasificación, clustering, ranking y generación](1.4-tipos-de-problema.ipynb)